## Metal Cutting (Column Generation)

Suppose we stock standard-length solid metal blocks that are used to craft android robots. The stock length is 19 meters.

We have an order for

* 12 lengths of 4m
* 15 lengths of 5m
* 22 lengths of 6m
 
How should we cut our standard length metal blocks to meet the demand and minimize the number of standard lengths we need to use?


### First Model

In [3]:
using JuMP, Gurobi

scale = 1

N = 16*scale  # upper bound on number of pipes needed

m = Model(solver = GurobiSolver(OutputFlag=0))

@variable(m, x[1:3,1:N] >= 0, Int)
@variable(m, z[1:N], Bin)
for j = 1:N
    @constraint(m, 4x[1,j] + 5x[2,j] + 6x[3,j] <= 19)
end
@constraint(m, sum(x[1,j] for j=1:N) >= 12*scale)
@constraint(m, sum(x[2,j] for j=1:N) >= 15*scale)
@constraint(m, sum(x[3,j] for j=1:N) >= 22*scale)
for j = 1:N
    @constraint(m, x[1,j] <= 4z[j])
    @constraint(m, x[2,j] <= 3z[j])
    @constraint(m, x[3,j] <= 3z[j])
end

# symmetry-breaking
for j = 1:N-1
    @constraint(m, z[j] >= z[j+1])
end

@objective(m, Min, sum(z[j] for j=1:N))

@time(solve(m))


Academic license - for non-commercial use only
  0.007502 seconds (127 allocations: 47.125 KiB)


:Optimal

### Second model (with column generation)

In [6]:
# all the columns:
A = [ 0 0 1 0 2 1 2 3 4
      0 1 0 2 1 2 2 1 0
      3 2 2 1 1 0 0 0 0 ]

scale = 100000

m = Model(solver = GurobiSolver(OutputFlag=0))

@variable(m, x[1:9] >= 0, Int)
@constraint(m, A*x .>= [12;15;22]*scale)
@objective(m, Min, sum(x))
@time(solve(m))


Academic license - for non-commercial use only
  0.001504 seconds (127 allocations: 16.063 KiB)


:Optimal